# Classification Track (Part A) - Hotel Booking Demand

**Problem statement.** Predict `is_canceled` - whether a booking will
eventually be cancelled - from the attributes known at the time the booking
is made. A hotel that can flag a booking as high-cancellation-risk before
arrival can overbook more safely and target offers at the guests who are most likely to cancel.

**Dataset.** `data/hotel_bookings.csv`, 119,390 bookings x 32 columns, from
Antonio, Almeida & Nunes (2019), *Hotel booking demand datasets*,
Data in Brief 22:41-49. Same file as the regression track; different target.

**Scope of this notebook.** Review 1 covers Classification Part A: the first
five algorithms (Logistic Regression, KNN, Naive Bayes, Decision Tree, SVM).

**Notebook map**:

| Section | Contents | Rubric |
|---|---|---|
| 1 | Load & audit | A1 |
| 2 | Exploratory data analysis | A2, A3 |
| 3 | Cleaning | B1 |
| 4 | Feature engineering | B3 |
| 5 | Encoding, scaling & splitting | B2 |
| 6 | Dimensionality reduction - PCA vs LDA | - |
| 7 | Five classification algorithms | D1 |
| 8 | Comparative evaluation | D2 |
| 9 | Cross-validation | (7.2) |
| 10 | Conclusion | - |


In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

DATA = "../data/hotel_bookings.csv"


ModuleNotFoundError: No module named 'seaborn'

## 1. Load & audit  *(A1)*

In [ ]:
df_raw = pd.read_csv(DATA)
print(f"rows: {df_raw.shape[0]:,}    columns: {df_raw.shape[1]}")
df_raw.head()


In [ ]:
audit = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "missing": df_raw.isna().sum(),
    "missing_%": (df_raw.isna().mean() * 100).round(2),
    "unique": df_raw.nunique(),
})
audit


In [ ]:
print("Target `is_canceled` (0 = kept, 1 = cancelled)")
counts = df_raw["is_canceled"].value_counts()
rate = df_raw["is_canceled"].mean() * 100
print(counts)
print(f"\ncancellation rate: {rate:.1f}%")


**Observation.** No column is fully empty, but four have large gaps: `company` (94.3%), `agent`
(13.7%), `country` (0.4%) and `children` (4 rows) - handled individually in a later section. The target is imbalanced at roughly 37% cancelled against 63%
kept, not extreme, but weighted F1 rather than plain accuracy is used
throughout so the majority class does not dominate the score.

## 2. Exploratory data analysis  *(A2, A3)*

### 2.1 Target distribution

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(data=df_raw, x="is_canceled", hue="is_canceled",
              palette=["#4C72B0", "#C44E52"], legend=False, ax=ax)
ax.set(title="Booking outcome", xlabel="is_canceled", ylabel="bookings")
ax.set_xticks([0, 1])
ax.set_xticklabels(["kept (0)", "cancelled (1)"])
for p in ax.patches:
    ax.annotate(f"{p.get_height():,}\n({p.get_height()/len(df_raw)*100:.1f}%)",
                (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom")
plt.tight_layout()
plt.show()


**Observation:** 63% of bookings are kept and 37% cancelled. That is
close enough to balanced that no resampling is
needed to get a usable model, but it is skewed enough that plain accuracy is
misleading - a model that always predicts "kept" already scores 63%. Weighted
F1 and the confusion matrix 8 are what actually judge these
models.

### 2.2 Distributions of the numeric features

In [ ]:
numeric_raw = df_raw.select_dtypes("number").columns.drop(["is_canceled"])
fig, axes = plt.subplots(5, 4, figsize=(15, 14))
for ax, col in zip(axes.ravel(), numeric_raw):
    sns.histplot(df_raw[col], bins=30, ax=ax, color="#4C72B0")
    ax.set(title=col, xlabel="", ylabel="")
for ax in axes.ravel()[len(numeric_raw):]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()


**Observation:** Most numeric columns are heavily right-skewed or zero
inflated - `previous_cancellations`, `booking_changes`,
`days_in_waiting_list`, `babies`, `required_car_parking_spaces`.
`lead_time` in particular has a long tail out past 400 days; Section 2.4
checks whether that tail carries cancellation signal.

### 2.3 Correlation structure

In [ ]:
corr_cols = list(numeric_raw) + ["is_canceled"]
corr = df_raw[corr_cols].corr()
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, cmap="vlag", center=0, annot=False, square=True,
            linewidths=.4, cbar_kws={"label": "Pearson r"}, ax=ax)
ax.set_title("Correlation matrix, numeric columns + is_canceled")
plt.tight_layout()
plt.show()

print(corr["is_canceled"].drop("is_canceled").sort_values(key=abs, ascending=False).round(3))


**Observation.** `lead_time` is the strongest single numeric
correlate of cancellation (r ~ 0.29): bookings made far in advance are more
likely to fall through. `total_of_special_requests` and
`required_car_parking_spaces` both correlate negatively - guests who commit
to specifics tend to follow through. `previous_cancellations` correlates
positively but weakly on its own, which is why Section 4 combines it with
`previous_bookings_not_canceled` into a rate rather than using the raw
count.

### 2.4 Feature-target relationships

In [ ]:
sample = df_raw.sample(6000, random_state=RANDOM_STATE)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))

sns.boxplot(data=sample, x="is_canceled", y="lead_time", ax=axes[0], color="#4C72B0")
axes[0].set(title="Lead time by outcome", xlabel="is_canceled", ylabel="lead time (days)")

sns.boxplot(data=sample, x="is_canceled", y="adr", ax=axes[1], color="#55A868")
axes[1].set(title="ADR by outcome", xlabel="is_canceled", ylabel="ADR (EUR)", ylim=(0, 300))

sns.countplot(data=sample, x="total_of_special_requests", hue="is_canceled",
              ax=axes[2])
axes[2].set(title="Special requests by outcome", xlabel="special requests", ylabel="bookings")
axes[2].legend(title="is_canceled", labels=["kept", "cancelled"])

plt.tight_layout()
plt.show()


**Observation.** Cancelled bookings have a visibly higher median lead
time and a biggerer upper whisker - the longer a booking sits on the books,
the more chances something changes. ADR barely separates the two groups,
consistent with Section 2.3's weak correlation. Special requests fall off
sharply for cancelled bookings: guests with zero special requests cancel
noticeably more often than guests with two or more, which can be interpreted as
engagement with the specific stay rather than a generic reservation.

### 2.5 Categorical drivers

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

for ax, col in zip(axes, ["deposit_type", "market_segment", "customer_type"]):
    rate = df_raw.groupby(col)["is_canceled"].mean().sort_values(ascending=False) * 100
    sns.barplot(x=rate.values, y=rate.index, ax=ax, color="#4C72B0")
    ax.set(title=f"Cancellation rate by {col}", xlabel="cancellation rate (%)", ylabel="")

plt.tight_layout()
plt.show()


**Observation.** `deposit_type` stands out as one of the stronger features. Interestingly, `Non Refund` bookings have a much higher cancellation rate than `No Deposit` bookings, which is the opposite of what we would normally expect. This seems to be a quirk of this dataset, so the model may pick up on this pattern even though it may not apply to hotels in general. `market_segment` also shows a higher cancellation rate for `Groups` compared to `Direct` and `Corporate`, while `customer_type` shows that `Transient` guests cancel more often than `Group` and `Contract` guests.